# Kouman IA — Notebook 1 : Pré-tokenisation & Shards 2048 (`Llama 3.2 1B`)

Ce notebook prépare **hors-ligne et de façon incrémentale (compatible sessions 1 h)** tous les corpus d'entraînement continu (CPT) avant le lancement du GPU pour **`meta-llama/Llama-3.2-1B`** (vocabulaire BPE 128 256 tokens) :

### Garanties d'ingénierie
1. **Zéro temps perdu pendant l'entraînement GPU** : tout est déjà tokenisé en blocs exacts de **`2048` tokens** avec `<|begin_of_text|>` / `<|end_of_text|>` (`meta-llama/Llama-3.2-1B`).
2. **Exclusion stricte du Nouchi (`nch`)** :
   - Aucun fichier `*_nch.parquet` n'est téléchargé.
   - Vérification stricte sur chaque ligne : `lang ∈ {"bci", "dyu", "fr", "en"}` (erreur levée si du `nch` apparaît).
3. **Mélange intra-shard exact (70 % langues nationales / 30 % internationales)** :
   - **35 % `bci`** (Baoulé : `cosmopedia-v2`, `fineweb-edu-dedup`, `finemath`, `open-thoughts`, `ivoiry-story-corpus`)
   - **35 % `dyu`** (Dioula : `cosmopedia-v2`, `fineweb-edu-dedup`)
   - **18 % `fr`** (Français : tous sous-ensembles + paires parallèles `FR ↔ BCI`)
   - **12 % `en`** (Anglais : `cosmopedia-v2`, `fineweb-edu-dedup`, `finemath`, `open-thoughts`)
4. **Reprise par micro-shard (~33,5M tokens / shard)** :
   - Si votre environnement s'arrête au bout de 1 h, un **watchdog à 46 min** sauvegarde le `manifeste_shards.json` sur Hugging Face et la session suivante reprend au fichier/row-group exact sans rien perdre.

## 1. Installation minimale & Authentification Hugging Face

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet --force-reinstall --no-cache-dir "transformers==4.57.3" "tokenizers==0.22.2" "huggingface_hub>=0.34.0,<1.0.0"
%pip install --quiet --upgrade-strategy only-if-needed "numpy>=2.0.0,<2.8.0" "pyarrow>=17.0.0" "tqdm"

In [ ]:
import getpass
import hashlib
import json
import os
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from huggingface_hub.errors import EntryNotFoundError, HfHubHTTPError, RepositoryNotFoundError
from tqdm.auto import tqdm
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "true"

HEURE_DEBUT_SESSION = time.monotonic()

# ═════════════════════════════════════════════════════════════════════════════
# CONFIGURATION UTILISATEUR (LLAMA 3.2 1B)
# ═════════════════════════════════════════════════════════════════════════════
JETON_HF = os.getenv("HF_TOKEN", "").strip()
if not JETON_HF and Path(".env").exists():
    for ligne_env in Path(".env").read_text(encoding="utf-8").splitlines():
        if ligne_env.strip().startswith("HF_TOKEN="):
            JETON_HF = ligne_env.split("=", 1)[1].strip().strip('"').strip("'")
            break
if not JETON_HF:
    JETON_HF = getpass.getpass("Jeton Hugging Face (lecture + écriture Tree-AI-lab + accès Llama 3.2) : ").strip()
if not JETON_HF:
    raise PermissionError("[ERREUR] Jeton HF obligatoire (dépôts privés + modèle gated meta-llama/Llama-3.2-1B).")
os.environ["HF_TOKEN"] = JETON_HF

MODELE_TOKENIZER = "meta-llama/Llama-3.2-1B"
DEPOT_CORPUS_SMOLLM = "Tree-AI-lab/smollm-corpus-multilingue"
DEPOT_CORPUS_IVOIRY = "Tree-AI-lab/ivoiry-story-corpus"
DEPOT_SORTIE_SHARDS = "Tree-AI-lab/kouman-llama32-1b-cpt-shards"
DEPOT_SHARDS_PRIVE = True

# Paramètres de découpage et de sécurité 1h
LONGUEUR_SEQUENCE = 2048
SEQUENCES_PAR_SHARD = 16_384     # 16 384 x 2048 = 33 554 432 tokens (~33,5M tokens / shard)
SHARDS_PAR_COMMIT_HF = 12        # Regroupe 12 shards + manifeste en 1 seul commit (anti-HTTP 429 HF)
DUREE_MAX_SESSION_MINUTES = 46.0 # Watchdog : arrêt propre avant la coupure à 60 min
GRAINE_ALEATOIRE = 42

# Proportions cibles strictes en tokens à l'intérieur de CHAQUE shard
QUOTAS_LANGUES = {
    "bci": 0.35,
    "dyu": 0.35,
    "fr": 0.18,
    "en": 0.12,
}
LANGUES_AUTORISEES = tuple(QUOTAS_LANGUES.keys())
LANGUES_INTERDITES = {"nch"}

# Époques cibles par langue : 3 passes sur bci/dyu, 1 passe sur fr/en
MAX_EPOQUES_PAR_LANGUE = {
    "bci": 3,
    "dyu": 3,
    "fr": 1,
    "en": 1,
}

# Jeu de validation figé par langue (exclu de l'entraînement via hachage SHA-256)
TAUX_VALIDATION = 0.005
MAX_SEQUENCES_EVAL_PAR_LANGUE = 256

DOSSIER_TRAVAIL = Path("travail_pretokenisation_llama32_1b")
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)
DOSSIER_SHARDS_LOCAUX = DOSSIER_TRAVAIL / "shards"
DOSSIER_SHARDS_LOCAUX.mkdir(parents=True, exist_ok=True)
DOSSIER_EVAL_LOCAL = DOSSIER_TRAVAIL / "eval"
DOSSIER_EVAL_LOCAL.mkdir(parents=True, exist_ok=True)

assert abs(sum(QUOTAS_LANGUES.values()) - 1.0) < 1e-9, "La somme des quotas doit valoir 1.0"
print(f"[OK] Configuration prête | Dépôt cible : {DEPOT_SORTIE_SHARDS}")
print(f"Répartition cible par shard : {QUOTAS_LANGUES} | Langues interdites : {LANGUES_INTERDITES}")

## 2. Chargement du Tokenizer Llama 3.2 1B & Contrats d'Erreurs

In [ ]:
class ErreurSchemaInvalide(ValueError):
    """Levée lorsqu'un fichier Parquet ne possède pas les colonnes attendues."""


class ErreurNouchiDetecte(ValueError):
    """Levée si une donnée Nouchi (nch) tente de passer le filtrage."""


class ErreurRepriseManifeste(RuntimeError):
    """Levée en cas d'incohérence ou d'échec réseau lors de la lecture du manifeste."""


try:
    tokenizer = AutoTokenizer.from_pretrained(MODELE_TOKENIZER, token=JETON_HF)
except Exception as erreur_tok:
    raise RuntimeError(
        f"[ERREUR] Impossible de charger le tokenizer '{MODELE_TOKENIZER}'. "
        "Vérifiez que votre compte Hugging Face a bien accepté la licence sur huggingface.co/meta-llama/Llama-3.2-1B."
    ) from erreur_tok

ID_BOS = tokenizer.bos_token_id
ID_EOS = tokenizer.eos_token_id
TAILLE_VOCAB = len(tokenizer)

if ID_BOS is None or ID_EOS is None:
    raise ValueError(f"[ERREUR] Tokens spéciaux invalides : bos={ID_BOS}, eos={ID_EOS}")

print(f"[OK] Tokenizer chargé : {MODELE_TOKENIZER}")
print(f"   • Vocabulaire : {TAILLE_VOCAB:,} tokens | BOS={ID_BOS} | EOS={ID_EOS}")

## 3. Découverte stricte des fichiers Parquet (Filtrage anti-Nouchi & Schémas)

In [ ]:
client_hf = HfApi(token=JETON_HF)

SOUS_ENSEMBLES_SMOLLM_AUTOS = ("cosmopedia-v2/", "fineweb-edu-dedup/", "finemath/", "open-thoughts/")


def lister_fichiers_autorises_smollm() -> list[str]:
    """Liste les Parquet SmolLM autorisés en excluant strictement Nouchi (nch) et python-edu."""
    tous_fichiers = client_hf.list_repo_files(
        repo_id=DEPOT_CORPUS_SMOLLM,
        repo_type="dataset",
        token=JETON_HF,
    )
    fichiers_retenus = []
    fichiers_nch_exclus = []

    for chemin in sorted(tous_fichiers):
        if not chemin.endswith(".parquet"):
            continue
        if not chemin.startswith(SOUS_ENSEMBLES_SMOLLM_AUTOS):
            continue
        nom_fichier = chemin.lower()
        if "_nch" in nom_fichier or "/nch" in nom_fichier:
            fichiers_nch_exclus.append(chemin)
            continue
        fichiers_retenus.append(chemin)

    if not fichiers_retenus:
        raise FileNotFoundError(
            f"[ERREUR] Aucun fichier Parquet autorisé trouvé dans {DEPOT_CORPUS_SMOLLM}."
        )

    for chemin in fichiers_retenus:
        if "nch" in Path(chemin).name.lower():
            raise ErreurNouchiDetecte(f"[ERREUR] Fichier Nouchi détecté dans la liste retenue : {chemin}")

    print(f"SmolLM : {len(fichiers_retenus)} fichiers Parquet retenus")
    print(f"Fichiers Nouchi (_nch) strictement exclus : {len(fichiers_nch_exclus)}")
    for exclu in fichiers_nch_exclus:
        print(f"   - exclu : {exclu}")
    return fichiers_retenus


def verifier_fichiers_ivoiry() -> list[str]:
    """Vérifie la présence des fichiers Parquet d'ivoiry-story-corpus."""
    tous_fichiers = client_hf.list_repo_files(
        repo_id=DEPOT_CORPUS_IVOIRY,
        repo_type="dataset",
        token=JETON_HF,
    )
    parquets = sorted(f for f in tous_fichiers if f.endswith(".parquet"))
    if not parquets:
        raise FileNotFoundError(f"[ERREUR] Aucun Parquet trouvé dans {DEPOT_CORPUS_IVOIRY}.")
    print(f"Ivoiry Story Corpus : {len(parquets)} fichier(s) Parquet vérifié(s)")
    return parquets


FICHIERS_SMOLLM = lister_fichiers_autorises_smollm()
FICHIERS_IVOIRY = verifier_fichiers_ivoiry()

## 4. Construction des Flux Monolingues & Validation sans Fallback

In [ ]:
def est_document_validation(cle_unique: str) -> bool:
    """Assigne de manière déterministe ~0,5 % des documents au jeu d'évaluation."""
    empreinte = hashlib.sha256(cle_unique.encode("utf-8")).digest()
    valeur = int.from_bytes(empreinte[:4], "big") % 10_000
    return valeur < int(TAUX_VALIDATION * 10_000)


def formater_texte_openthoughts(probleme: str, pensee: str, solution: str, langue: str) -> str:
    """Assemble un raisonnement OpenThoughts complet sans couper la chaîne de pensée."""
    p = (probleme or "").strip()
    t = (pensee or "").strip()
    s = (solution or "").strip()
    if not p or not s:
        return ""
    if langue == "fr":
        etiquettes = ("Problème :", "Réponse :")
    elif langue == "bci":
        etiquettes = ("Ndɛ :", "Tɛlɛ :")
    else:
        etiquettes = ("Problem:", "Solution:")
    if t:
        return f"{etiquettes[0]}\n{p}\n\n<think>\n{t}\n</think>\n\n{etiquettes[1]}\n{s}"
    return f"{etiquettes[0]}\n{p}\n\n{etiquettes[1]}\n{s}"


def extraire_documents_row_group(
    fichier_pq: pq.ParquetFile,
    indice_rg: int,
    depot_hf: str,
    chemin_fichier: str,
) -> dict[str, list[tuple[str, str]]]:
    """Lit un seul row-group PyArrow (faible RAM) et extrait (cle_id, texte) par langue autorisée."""
    colonnes = set(fichier_pq.schema_arrow.names)
    documents_par_langue: dict[str, list[tuple[str, str]]] = {lg: [] for lg in LANGUES_AUTORISEES}

    if depot_hf == DEPOT_CORPUS_IVOIRY:
        attendues = {"wikipedia_texte", "wikipedia_texte_bci"}
        if not attendues.issubset(colonnes):
            raise ErreurSchemaInvalide(
                f"[ERREUR] Schéma inattendu dans {depot_hf}/{chemin_fichier} : colonnes trouvées={sorted(colonnes)}"
            )
        a_titre = "wikipedia_titre" in colonnes
        cols_lecture = ["wikipedia_texte", "wikipedia_texte_bci"] + (["wikipedia_titre"] if a_titre else [])
        dico = fichier_pq.read_row_group(indice_rg, columns=cols_lecture).to_pydict()
        nb = len(dico["wikipedia_texte"])
        for i in range(nb):
            titre = (dico["wikipedia_titre"][i] or "").strip() if a_titre else ""
            fr_txt = (dico["wikipedia_texte"][i] or "").strip()
            bci_txt = (dico["wikipedia_texte_bci"][i] or "").strip()
            cle_base = f"ivoiry:{chemin_fichier}:rg{indice_rg}:{i}"
            if bci_txt:
                corps_bci = f"{titre}\n\n{bci_txt}" if titre else bci_txt
                documents_par_langue["bci"].append((cle_base + ":bci", corps_bci))
            if fr_txt and bci_txt:
                paire_fr_bci = f"[Français]\n{fr_txt}\n\n[Baoulé]\n{bci_txt}"
                paire_bci_fr = f"[Baoulé]\n{bci_txt}\n\n[Français]\n{fr_txt}"
                documents_par_langue["bci"].append((cle_base + ":paire1", paire_fr_bci))
                documents_par_langue["fr"].append((cle_base + ":paire2", paire_bci_fr))
            elif fr_txt:
                documents_par_langue["fr"].append((cle_base + ":fr", fr_txt))
        return documents_par_langue

    if {"lang", "text"}.issubset(colonnes):
        dico = fichier_pq.read_row_group(indice_rg, columns=["lang", "text"]).to_pydict()
        for i, (lg_brute, txt_brut) in enumerate(zip(dico["lang"], dico["text"])):
            lg = (lg_brute or "").strip().lower()
            if lg in LANGUES_INTERDITES:
                raise ErreurNouchiDetecte(
                    f"[ERREUR] Ligne Nouchi ('{lg}') trouvée dans {chemin_fichier} (row_group={indice_rg}, index={i}) !"
                )
            if lg not in documents_par_langue:
                raise ErreurSchemaInvalide(
                    f"[ERREUR] Code langue inconnu '{lg}' dans {chemin_fichier} (attendu: {LANGUES_AUTORISEES})"
                )
            txt = (txt_brut or "").strip()
            if txt:
                documents_par_langue[lg].append((f"{chemin_fichier}:rg{indice_rg}:{i}", txt))
        return documents_par_langue

    attendues_ot = {
        "problem_en", "thinking_en", "solution_en",
        "problem_fr", "thinking_fr", "solution_fr",
        "problem_bci", "thinking_bci", "solution_bci",
    }
    if attendues_ot.issubset(colonnes):
        dico = fichier_pq.read_row_group(indice_rg, columns=sorted(attendues_ot)).to_pydict()
        nb = len(dico["problem_en"])
        for i in range(nb):
            cle_base = f"ot:{chemin_fichier}:rg{indice_rg}:{i}"
            for lg in ("bci", "fr", "en"):
                txt = formater_texte_openthoughts(
                    dico[f"problem_{lg}"][i],
                    dico[f"thinking_{lg}"][i],
                    dico[f"solution_{lg}"][i],
                    lg,
                )
                if txt:
                    documents_par_langue[lg].append((f"{cle_base}:{lg}", txt))
        return documents_par_langue

    raise ErreurSchemaInvalide(
        f"[ERREUR] Schéma inattendu dans {chemin_fichier} : colonnes={sorted(colonnes)}"
    )

## 5. Indexation des sources par langue & Reprise d'état sur Hugging Face

In [ ]:
def construire_catalogue_sources() -> dict[str, list[tuple[str, str]]]:
    """Regroupe les fichiers sources pertinents par langue cible."""
    catalogue: dict[str, list[tuple[str, str]]] = {lg: [] for lg in LANGUES_AUTORISEES}

    for f in FICHIERS_IVOIRY:
        catalogue["bci"].append((DEPOT_CORPUS_IVOIRY, f))
        catalogue["fr"].append((DEPOT_CORPUS_IVOIRY, f))

    for f in FICHIERS_SMOLLM:
        nom = Path(f).name.lower()
        if f.startswith("finemath/") or f.startswith("open-thoughts/"):
            for lg in ("bci", "fr", "en"):
                catalogue[lg].append((DEPOT_CORPUS_SMOLLM, f))
        else:
            trouve = False
            for lg in LANGUES_AUTORISEES:
                if f"_{lg}" in nom:
                    catalogue[lg].append((DEPOT_CORPUS_SMOLLM, f))
                    trouve = True
                    break
            if not trouve:
                raise ErreurSchemaInvalide(f"[ERREUR] Impossible d'inférer la langue du fichier SmolLM : {f}")

    for lg, items in catalogue.items():
        if not items:
            raise ValueError(f"[ERREUR] Aucun fichier source trouvé pour la langue '{lg}'.")
        print(f"   • {lg} : {len(items)} fichier(s) source(s)")
    return catalogue


def initialiser_ou_charger_manifeste(catalogue: dict[str, list[tuple[str, str]]]) -> dict:
    """Charge le manifeste depuis HF s'il existe, sinon crée un état vierge légitime."""
    client_hf.create_repo(
        repo_id=DEPOT_SORTIE_SHARDS,
        repo_type="dataset",
        private=DEPOT_SHARDS_PRIVE,
        exist_ok=True,
    )
    try:
        fichiers_depot = client_hf.list_repo_files(
            repo_id=DEPOT_SORTIE_SHARDS,
            repo_type="dataset",
            token=JETON_HF,
        )
    except RepositoryNotFoundError as err:
        raise ErreurRepriseManifeste(f"[ERREUR] Dépôt cible inaccessible : {DEPOT_SORTIE_SHARDS}") from err

    if "manifeste_shards.json" not in fichiers_depot:
        print("Aucun manifeste existant sur HF : démarrage d'une nouvelle campagne de sharding Llama 3.2 1B.")
        return {
            "version": 1,
            "modele_tokenizer": MODELE_TOKENIZER,
            "longueur_sequence": LONGUEUR_SEQUENCE,
            "sequences_par_shard": SEQUENCES_PAR_SHARD,
            "quotas_langues": QUOTAS_LANGUES,
            "langues_interdites": list(LANGUES_INTERDITES),
            "termine": False,
            "eval_genere": ("eval/eval_multilingue.parquet" in fichiers_depot),
            "prochain_shard": 0,
            "shards_publies": [],
            "tokens_totaux_publies": 0,
            "tokens_par_langue": {lg: 0 for lg in LANGUES_AUTORISEES},
            "curseurs": {
                lg: {"indice_fichier": 0, "indice_rg": 0, "epoque": 0}
                for lg in LANGUES_AUTORISEES
            },
        }

    chemin_manifeste = hf_hub_download(
        repo_id=DEPOT_SORTIE_SHARDS,
        filename="manifeste_shards.json",
        repo_type="dataset",
        token=JETON_HF,
        force_download=True,
    )
    manifeste = json.loads(Path(chemin_manifeste).read_text(encoding="utf-8"))
    if manifeste.get("modele_tokenizer") != MODELE_TOKENIZER:
        raise ErreurRepriseManifeste(
            f"[ERREUR] Incompatibilité de tokenizer dans le manifeste : {manifeste.get('modele_tokenizer')} != {MODELE_TOKENIZER}"
        )
    if manifeste.get("longueur_sequence") != LONGUEUR_SEQUENCE:
        raise ErreurRepriseManifeste("[ERREUR] Incompatibilité de LONGUEUR_SEQUENCE avec le manifeste existant.")
    print(
        f"Manifeste repris depuis HF : {manifeste['prochain_shard']} shard(s) déjà publiés "
        f"({manifeste['tokens_totaux_publies']:,} tokens)."
    )
    return manifeste


CATALOGUE_SOURCES = construire_catalogue_sources()
manifeste = initialiser_ou_charger_manifeste(CATALOGUE_SOURCES)

## 6. Moteur d'Empaquetage par Quotas Exacts & Watchdog 46 min

In [ ]:
def calculer_cibles_sequences_shard() -> dict[str, int]:
    """Calcule le nombre exact de séquences de 2048 tokens par langue dans un shard."""
    cibles = {}
    somme = 0
    langues = list(QUOTAS_LANGUES.keys())
    for lg in langues[:-1]:
        n = int(round(SEQUENCES_PAR_SHARD * QUOTAS_LANGUES[lg]))
        cibles[lg] = n
        somme += n
    cibles[langues[-1]] = SEQUENCES_PAR_SHARD - somme
    assert sum(cibles.values()) == SEQUENCES_PAR_SHARD
    return cibles


CIBLES_SHARD = calculer_cibles_sequences_shard()
print("Séquences de 2048 tokens par shard :", CIBLES_SHARD)


class ReserveTokensLangue:
    """Gère le flux paresseux par row-group Parquet et l'empaquetage 2048 sans padding."""

    def __init__(self, langue: str, sources: list[tuple[str, str]], etat_initial: dict):
        self.langue = langue
        self.sources = sources
        self.indice_fichier = int(etat_initial["indice_fichier"])
        self.indice_rg = int(etat_initial.get("indice_rg", 0))
        self.epoque = int(etat_initial["epoque"])
        self.max_epoques = MAX_EPOQUES_PAR_LANGUE[langue]
        self.tampon_tokens: list[int] = []
        self.blocs_train_prets: list[list[int]] = []
        self.blocs_eval_prets: list[list[int]] = []
        self.tampon_eval: list[int] = []
        self.epuisee = self.epoque >= self.max_epoques
        self._handle_pq: pq.ParquetFile | None = None
        self._cle_pq_ouvert: tuple[str, str] | None = None

    def _empaqueter_liste(self, docs_textes: list[str], tampon: list[int], destination: list[list[int]], taille_lot_tok: int = 256):
        """Tokenise par lots rapides et découpe en blocs exacts de LONGUEUR_SEQUENCE."""
        for debut in range(0, len(docs_textes), taille_lot_tok):
            sous_lot = docs_textes[debut:debut + taille_lot_tok]
            encodage = tokenizer(
                sous_lot,
                add_special_tokens=False,
                truncation=False,
                return_attention_mask=False,
            )
            for ids in encodage["input_ids"]:
                if not ids:
                    continue
                tampon.append(ID_BOS)
                tampon.extend(ids)
                tampon.append(ID_EOS)
            while len(tampon) >= LONGUEUR_SEQUENCE:
                destination.append(tampon[:LONGUEUR_SEQUENCE])
                del tampon[:LONGUEUR_SEQUENCE]

    def _ouvrir_fichier_courant(self) -> tuple[pq.ParquetFile, str, str] | None:
        """Ouvre le fichier Parquet courant ou avance à l'époque suivante si nécessaire."""
        while not self.epuisee:
            if self.indice_fichier >= len(self.sources):
                self.epoque += 1
                self.indice_fichier = 0
                self.indice_rg = 0
                self._handle_pq = None
                self._cle_pq_ouvert = None
                if self.epoque >= self.max_epoques:
                    self.epuisee = True
                    return None

            depot, chemin_fichier = self.sources[self.indice_fichier]
            cle = (depot, chemin_fichier)
            if self._handle_pq is None or self._cle_pq_ouvert != cle:
                chemin_local = hf_hub_download(
                    repo_id=depot,
                    filename=chemin_fichier,
                    repo_type="dataset",
                    token=JETON_HF,
                )
                self._handle_pq = pq.ParquetFile(chemin_local)
                self._cle_pq_ouvert = cle
                print(
                    f"[{self.langue} | Époque {self.epoque + 1}/{self.max_epoques}] "
                    f"Ouverture {depot}/{chemin_fichier} ({self._handle_pq.num_row_groups} row-groups, reprise rg={self.indice_rg})"
                )

            if self.indice_rg >= self._handle_pq.num_row_groups:
                self.indice_fichier += 1
                self.indice_rg = 0
                self._handle_pq = None
                self._cle_pq_ouvert = None
                continue
            return self._handle_pq, depot, chemin_fichier
        return None

    def charger_fichier_suivant(self, collecter_eval: bool) -> bool:
        """Lit un seul row-group Parquet (mémoire bornée) et alimente les tampons train/eval."""
        ouvert = self._ouvrir_fichier_courant()
        if ouvert is None:
            return False

        fichier_pq, depot, chemin_fichier = ouvert
        docs_par_langue = extraire_documents_row_group(fichier_pq, self.indice_rg, depot, chemin_fichier)
        rg_lu = self.indice_rg
        self.indice_rg += 1

        docs_bruts = docs_par_langue[self.langue]
        if not docs_bruts:
            return self.charger_fichier_suivant(collecter_eval)

        docs_train = []
        docs_eval = []
        for cle, texte in docs_bruts:
            if collecter_eval and self.epoque == 0 and est_document_validation(cle):
                docs_eval.append(texte)
            else:
                docs_train.append(texte)

        rng_local = random.Random(GRAINE_ALEATOIRE + self.epoque * 10_000 + self.indice_fichier * 100 + rg_lu)
        rng_local.shuffle(docs_train)

        if docs_eval:
            self._empaqueter_liste(docs_eval, self.tampon_eval, self.blocs_eval_prets)
        if docs_train:
            self._empaqueter_liste(docs_train, self.tampon_tokens, self.blocs_train_prets)
        return True

    def prelever_sequences(self, nb_souhaite: int, collecter_eval: bool) -> list[list[int]]:
        """Fournit exactement nb_souhaite séquences (ou moins si la langue est définitivement épuisée)."""
        while len(self.blocs_train_prets) < nb_souhaite and not self.epuisee:
            if not self.charger_fichier_suivant(collecter_eval):
                break
        prise = min(nb_souhaite, len(self.blocs_train_prets))
        resultat = self.blocs_train_prets[:prise]
        del self.blocs_train_prets[:prise]
        return resultat


def ecrire_et_pousser_eval(reserves: dict[str, ReserveTokensLangue]) -> None:
    """Construit et pousse le fichier d'évaluation figé eval_multilingue.parquet."""
    lignes_ids = []
    lignes_lang = []
    for lg, res in reserves.items():
        tentatives = 0
        while len(res.blocs_eval_prets) < MAX_SEQUENCES_EVAL_PAR_LANGUE and not res.epuisee and tentatives < 60:
            res.charger_fichier_suivant(collecter_eval=True)
            tentatives += 1
        blocs = res.blocs_eval_prets[:MAX_SEQUENCES_EVAL_PAR_LANGUE]
        if not blocs:
            raise ValueError(f"[ERREUR] Jeu d'évaluation vide pour la langue '{lg}' !")
        for bloc in blocs:
            assert len(bloc) == LONGUEUR_SEQUENCE
            lignes_ids.append(bloc)
            lignes_lang.append(lg)

    table_eval = pa.table(
        {
            "input_ids": pa.array(lignes_ids, type=pa.list_(pa.int32(), LONGUEUR_SEQUENCE)),
            "lang": pa.array(lignes_lang, type=pa.string()),
        }
    )
    chemin_eval = DOSSIER_EVAL_LOCAL / "eval_multilingue.parquet"
    pq.write_table(table_eval, chemin_eval, compression="zstd")
    client_hf.upload_file(
        path_or_fileobj=str(chemin_eval),
        path_in_repo="eval/eval_multilingue.parquet",
        repo_id=DEPOT_SORTIE_SHARDS,
        repo_type="dataset",
        commit_message="feat: split évaluation multilingue figé Llama 3.2 1B (bci, dyu, fr, en)",
    )
    print(f"Split évaluation publié : {len(lignes_ids)} séquences ({chemin_eval.stat().st_size / 1e6:.2f} Mo)")


def ecrire_manifeste_local(manifeste_courant: dict) -> Path:
    manifeste_courant["derniere_maj_utc"] = datetime.now(timezone.utc).isoformat()
    chemin_manifeste = DOSSIER_TRAVAIL / "manifeste_shards.json"
    chemin_manifeste.write_text(json.dumps(manifeste_courant, indent=2, ensure_ascii=False), encoding="utf-8")
    return chemin_manifeste


def pousser_lot_atomique_hf(
    lots_shards_locaux: list[tuple[Path, str]],
    manifeste_courant: dict,
    msg_commit: str,
) -> None:
    """Pousse en 1 SEUL commit atomique tous les shards en attente + manifeste_shards.json."""
    if not lots_shards_locaux:
        return
    chemin_manifeste = ecrire_manifeste_local(manifeste_courant)
    operations = [
        CommitOperationAdd(path_or_fileobj=str(chemin_local), path_in_repo=chemin_repo)
        for chemin_local, chemin_repo in lots_shards_locaux
    ]
    operations.append(
        CommitOperationAdd(path_or_fileobj=str(chemin_manifeste), path_in_repo="manifeste_shards.json")
    )
    try:
        client_hf.create_commit(
            repo_id=DEPOT_SORTIE_SHARDS,
            repo_type="dataset",
            operations=operations,
            commit_message=msg_commit,
        )
    except HfHubHTTPError as err_hf:
        if getattr(err_hf.response, "status_code", None) == 429:
            raise RuntimeError(
                "Limite de commits Hugging Face (128/h) atteinte. "
                f"{len(lots_shards_locaux)} shard(s) restent prêts en local dans {DOSSIER_SHARDS_LOCAUX}."
            ) from err_hf
        raise

    for chemin_local, _ in lots_shards_locaux:
        chemin_local.unlink(missing_ok=True)
    lots_shards_locaux.clear()


def temps_ecoule_minutes() -> float:
    return (time.monotonic() - HEURE_DEBUT_SESSION) / 60.0

## 7. Exécution du Sharding Incrémental & Commit Atomique Groupé

In [ ]:
if manifeste["termine"]:
    print("Tous les shards planifiés Llama 3.2 1B ont déjà été générés et publiés !")
else:
    reserves = {
        lg: ReserveTokensLangue(lg, CATALOGUE_SOURCES[lg], manifeste["curseurs"][lg])
        for lg in LANGUES_AUTORISEES
    }

    if not manifeste["eval_genere"]:
        print("Génération initiale du split d'évaluation figé...")
        ecrire_et_pousser_eval(reserves)
        manifeste["eval_genere"] = True
        chemin_eval_manifeste = ecrire_manifeste_local(manifeste)
        client_hf.upload_file(
            path_or_fileobj=str(chemin_eval_manifeste),
            path_in_repo="manifeste_shards.json",
            repo_id=DEPOT_SORTIE_SHARDS,
            repo_type="dataset",
            commit_message="chore: enregistrement eval_genere",
        )

    rng_global = random.Random(GRAINE_ALEATOIRE + manifeste["prochain_shard"])
    lots_en_attente_commit: list[tuple[Path, str]] = []

    try:
        while True:
            if temps_ecoule_minutes() >= DUREE_MAX_SESSION_MINUTES:
                print(
                    f"Watchdog déclenché ({temps_ecoule_minutes():.1f} min >= {DUREE_MAX_SESSION_MINUTES} min). "
                    "Push groupé avant coupure de session !"
                )
                break

            if reserves["bci"].epuisee and reserves["dyu"].epuisee:
                print("Les langues principales bci et dyu ont complété toutes leurs époques cibles !")
                manifeste["termine"] = True
                break

            for lg_support in ("fr", "en"):
                if reserves[lg_support].epuisee and not (reserves["bci"].epuisee and reserves["dyu"].epuisee):
                    print(f"Recyclage contrôlé de '{lg_support}' pour maintenir exactement 30 % FR+EN.")
                    reserves[lg_support].epuisee = False
                    reserves[lg_support].indice_fichier = 0
                    reserves[lg_support].indice_rg = 0

            paires_shard: list[tuple[list[int], str]] = []
            comptes_shard = {}

            for lg, cible_n in CIBLES_SHARD.items():
                seqs = reserves[lg].prelever_sequences(cible_n, collecter_eval=False)
                comptes_shard[lg] = len(seqs)
                for s in seqs:
                    if lg in LANGUES_INTERDITES:
                        raise ErreurNouchiDetecte(f"[ERREUR] Langue interdite détectée avant écriture : {lg}")
                    paires_shard.append((s, lg))

            if len(paires_shard) < 512:
                print(f"[INFO] Reliquat final trop petit ({len(paires_shard)} séquences) — fin du sharding.")
                manifeste["termine"] = True
                break

            rng_global.shuffle(paires_shard)

            tableau_ids = [p[0] for p in paires_shard]
            tableau_lang = [p[1] for p in paires_shard]

            assert "nch" not in set(tableau_lang), "ERREUR FATALE : 'nch' présent dans un shard !"
            assert all(len(seq) == LONGUEUR_SEQUENCE for seq in tableau_ids), "Longueur de séquence invalide !"

            nom_shard = f"train-{manifeste['prochain_shard']:05d}.parquet"
            chemin_shard_local = DOSSIER_SHARDS_LOCAUX / nom_shard
            table_shard = pa.table(
                {
                    "input_ids": pa.array(tableau_ids, type=pa.list_(pa.int32(), LONGUEUR_SEQUENCE)),
                    "lang": pa.array(tableau_lang, type=pa.string()),
                }
            )
            pq.write_table(table_shard, chemin_shard_local, compression="zstd")

            chemin_repo_shard = f"train/{nom_shard}"
            lots_en_attente_commit.append((chemin_shard_local, chemin_repo_shard))

            tokens_shard = len(paires_shard) * LONGUEUR_SEQUENCE
            manifeste["shards_publies"].append(
                {
                    "fichier": chemin_repo_shard,
                    "nb_sequences": len(paires_shard),
                    "nb_tokens": tokens_shard,
                    "repartition": comptes_shard,
                }
            )
            manifeste["prochain_shard"] += 1
            manifeste["tokens_totaux_publies"] += tokens_shard
            for lg, n_seq in comptes_shard.items():
                manifeste["tokens_par_langue"][lg] += n_seq * LONGUEUR_SEQUENCE
                manifeste["curseurs"][lg] = {
                    "indice_fichier": reserves[lg].indice_fichier,
                    "indice_rg": reserves[lg].indice_rg,
                    "epoque": reserves[lg].epoque,
                }

            print(
                f"{nom_shard} prêt en local ({temps_ecoule_minutes():.1f} min) | "
                f"Lot={len(lots_en_attente_commit)}/{SHARDS_PAR_COMMIT_HF} | "
                f"Cumul={manifeste['tokens_totaux_publies'] / 1e9:.3f}B tokens"
            )

            if len(lots_en_attente_commit) >= SHARDS_PAR_COMMIT_HF:
                premier = Path(lots_en_attente_commit[0][1]).name
                dernier = Path(lots_en_attente_commit[-1][1]).name
                pousser_lot_atomique_hf(
                    lots_en_attente_commit,
                    manifeste,
                    f"feat: lot atomique {premier}..{dernier} ({manifeste['tokens_totaux_publies'] / 1e9:.3f}B tokens)",
                )
                print(f"Lot {premier}..{dernier} poussé sur HF en 1 commit atomique !")
    finally:
        if lots_en_attente_commit:
            premier = Path(lots_en_attente_commit[0][1]).name
            dernier = Path(lots_en_attente_commit[-1][1]).name
            print(f"Flush final de {len(lots_en_attente_commit)} shard(s) restant(s) ({premier}..{dernier})...")
            pousser_lot_atomique_hf(
                lots_en_attente_commit,
                manifeste,
                f"feat: flush fin de session {premier}..{dernier}",
            )
        elif manifeste["termine"]:
            chemin_m = ecrire_manifeste_local(manifeste)
            client_hf.upload_file(
                path_or_fileobj=str(chemin_m),
                path_in_repo="manifeste_shards.json",
                repo_id=DEPOT_SORTIE_SHARDS,
                repo_type="dataset",
                commit_message="chore: marquage termine=True",
            )

    print("Session terminée proprement. Relancez ce notebook si manifeste['termine'] est False.")